# CNN_5_7ch_Exp17B by Hard Voting Ensemble

# 1.Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

# 2. Setting

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")

EXP17_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_Focal_seed44",
}
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp17B_HV_Ensemble"
assert RESULT_DIR not in EXP17_DIRS.values()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
SEEDS = [42, 43, 44]
TIE_SEED = 42

# 3.โหลด CSV และตรวจความตรงกัน

In [ ]:
saved = {
    seed: pd.read_csv(EXP17_DIRS[seed] / "val_predictions.csv")
    for seed in SEEDS
}
base = saved[42]
assert len(base) == 2878

for seed, df in saved.items():
    assert len(df) == len(base)
    for col in KEY_COLS:
        assert (df[col].astype(str).to_numpy() == base[col].astype(str).to_numpy()).all(), f"Seed {seed}: {col} ไม่ตรงกัน"

    prob = df[PROB_COLS].to_numpy()
    pred = np.array(STAGES)[prob.argmax(axis=1)]

    assert np.isfinite(prob).all()
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-3)
    assert (pred == df["predicted"].to_numpy()).all()
    assert np.allclose(df["confidence"], prob.max(axis=1), atol=1e-5)
    assert (df["correct"].astype(str).str.upper().to_numpy() == np.where(df["actual"] == df["predicted"], "TRUE", "FALSE")).all()

y_val = base["actual"].map(dict(zip(STAGES, LABELS))).to_numpy()
assert not pd.isna(y_val).any()
y_val = y_val.astype(int)
probs = {seed: df[PROB_COLS].to_numpy() for seed, df in saved.items()}

print("ตัวอย่างตรงกัน:", len(y_val), "sequences,", base["subject"].nunique(), "subjects")

# 4.Hard Voting

In [ ]:
votes = np.stack([probs[s].argmax(axis=1) for s in SEEDS])
counts = np.stack([(votes == k).sum(axis=0) for k in LABELS], axis=1)

is_tie = (counts == counts.max(axis=1, keepdims=True)).sum(axis=1) > 1
hard_pred = counts.argmax(axis=1)
hard_pred[is_tie] = votes[SEEDS.index(TIE_SEED)][is_tie]

print("Hard Voting เสมอ:", int(is_tie.sum()),"ตัวอย่าง (ใช้คำทำนายของ Seed", TIE_SEED, ")")

# 5.คำนวณ metric แสดงผล เปรียบเทียบ และบันทึก

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES, "precision": precision, "recall": recall,
        "f1_score": f1, "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def show_result(name, y_pred):
    overall, per_class = calc_metrics(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred, labels=LABELS)
    cm_norm = confusion_matrix(y_val, y_pred, labels=LABELS, normalize="true")

    print("=====", name, "(VAL) =====")
    display(pd.DataFrame([overall]).round(4))
    display(per_class.round(4))
    display(stage_confusion(cm).round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Confusion matrix (count)")
    ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1].set_title("Confusion matrix (normalized by actual)")
    plt.suptitle(name)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{name}_val_confusion_matrix.png", dpi=150)
    plt.show()

    pd.DataFrame([overall]).to_csv(RESULT_DIR / f"{name}_val_overall_metrics.csv", index=False)
    per_class.to_csv(RESULT_DIR / f"{name}_val_per_class_metrics.csv", index=False)
    stage_confusion(cm).to_csv(RESULT_DIR / f"{name}_val_per_stage_confusion.csv", index=False)
    pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{name}_val_confusion_matrix.csv")
    return overall, per_class


rows, f1_rows = {}, {}
for seed in SEEDS:
    overall, per_class = calc_metrics(y_val, probs[seed].argmax(axis=1))
    rows[f"Exp17B Seed {seed}"] = overall
    f1_rows[f"Exp17B Seed {seed}"] = per_class["f1_score"].to_numpy()
display(pd.DataFrame(rows).T.round(4))

hard_overall, hard_pc = show_result("hard_voting", hard_pred)

rows["Hard Voting"], f1_rows["Hard Voting"] = hard_overall, hard_pc["f1_score"].to_numpy()

comparison = pd.DataFrame(rows).T
comparison.index.name = "method"
f1_comparison = pd.DataFrame(f1_rows, index=STAGES).T
f1_comparison.index.name = "method"

print("เปรียบเทียบ Validation (overall)")
display(comparison.round(4))
print("เปรียบเทียบ F1 รายคลาส")
display(f1_comparison.round(4))

comparison.to_csv(RESULT_DIR / "val_comparison_overall.csv")
f1_comparison.to_csv(RESULT_DIR / "val_comparison_f1_per_class.csv")

pred_table = base[["subject", "epoch_start", "epoch_end", "actual"]].copy()
for seed in SEEDS:
    pred_table[f"pred_seed{seed}"] = np.array(STAGES)[probs[seed].argmax(axis=1)]
pred_table["hard_tie"] = is_tie
pred_table["pred_hard"] = np.array(STAGES)[hard_pred]

pred_table.to_csv(RESULT_DIR / "val_ensemble_predictions.csv", index=False)